# Introducción al Prompt Engineering con NVIDIA NIM

El *prompt engineering* es el proceso de diseñar y evaluar instrucciones para
obtener respuestas útiles, consistentes y verificables de un modelo de lenguaje.
En este notebook utilizaremos **NVIDIA Nemotron 3 Super** mediante el endpoint
alojado de **NVIDIA NIM**, compatible con la API de OpenAI.

Antes de comenzar, crea una API key en
[NVIDIA Build](https://build.nvidia.com/settings/api-keys) y guárdala en un
archivo `.env`:

```text
NVIDIA_API_KEY="nvapi-..."
```

> Nunca escribas la API key directamente en el notebook ni la publiques en GitHub.

**Instalación** — ejecutar una sola vez en Colab:

In [1]:
!pip install -q --upgrade openai python-dotenv


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuración inicial

Creamos un cliente compatible con OpenAI, apuntándolo al endpoint de NVIDIA.
La función `get_completion` centraliza los parámetros que usaremos durante el
laboratorio. Desactivamos el razonamiento interno para concentrarnos en el efecto
visible de cada prompt.

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    raise ValueError("No se encontró NVIDIA_API_KEY. Configúrala en el archivo .env")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"


def get_completion(
    prompt: str,
    *,
    system: str = "You are a helpful assistant.",
    temperature: float = 1.0,
    max_tokens: int = 1024,
) -> str:
    """Realiza una llamada de un solo turno a NVIDIA NIM."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        top_p=0.95,
        max_tokens=max_tokens,
        extra_body={
            "chat_template_kwargs": {"enable_thinking": False}
        },
    )
    return response.choices[0].message.content

## Ejercicio 1: tokens y consumo de contexto

Los modelos no procesan texto carácter a carácter, sino mediante **tokens**.
NVIDIA NIM reporta el uso real de tokens después de cada solicitud mediante
`response.usage`.

El conteo de entrada incluye tanto el texto como la estructura de los mensajes
(`system`, `user` y plantilla de chat); por eso no representa únicamente los
tokens del párrafo.

**Ejercicio:**

1. Ejecuta la celda y observa `prompt_tokens`.
2. Cambia el texto por español, código o emojis.
3. Compara caracteres por token.
4. Explica por qué esta razón es solo una aproximación.

In [4]:
text = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter."
)

token_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "Reply only with OK."},
        {"role": "user", "content": text},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

prompt_tokens = token_response.usage.prompt_tokens
print(f"Tokens de entrada de la solicitud: {prompt_tokens}")
print(f"Caracteres del texto: {len(text)}")
print(f"Razón aproximada: {len(text) / prompt_tokens:.1f} caracteres/token")

Tokens de entrada de la solicitud: 113
Caracteres del texto: 423
Razón aproximada: 3.7 caracteres/token


## Ejercicio 2: validar la configuración de la API

Verifica que la clave y el endpoint funcionan correctamente con una solicitud
simple.

- **Entrada:** `oh say can you see`
- **Resultado esperado:** el modelo debería continuar o reconocer el verso.

In [5]:
prompt = "```\noh say can you see\n```"
response = get_completion(prompt)
print(response)

It looks like you've quoted the opening line of "The Star-Spangled Banner," the national anthem of the United States. The full line is:

> *"Oh! say can you see, by the dawn's early light..."*

Would you like the full lyrics, some history about the song, or help with something else related to it? 😊


## Ejercicio 3: fabricaciones o alucinaciones

Los LLM pueden producir información plausible sobre hechos inexistentes. Un
prompt mejor puede reducir este riesgo, pero no garantiza factualidad.

**Ejercicio:**

- Ejecuta el prompt sobre el evento ficticio.
- Añade: `If this event is fictional or unverifiable, say so explicitly.`
- Compara las respuestas.
- Discute por qué cambiar la temperatura no sustituye la verificación con fuentes.

In [6]:
prompt = """
Generate a lesson plan on the Martian War of 2076. If this event is fictional or unverifiable, say so explicitly
""".strip()

response = get_completion(prompt)
print(response)

Certainly. Since the **Martian War of 2076** is not a historical event that has occurred and there is no verifiable record of such a conflict in scientific, political, or historical sources, I will explicitly state that it is **fictional or unverifiable** before proceeding.

---

**Explicit Disclaimer:**  
> The Martian War of 2076 is a fictional event. No such war occurred in reality as of 2024. Humanity has not established permanent settlements on Mars, nor has any armed conflict taken place there. All references to the "Martian War of 2076" are speculative, imaginative, or drawn from science fiction narratives (e.g., novels, films, or games). This lesson plan is designed as a **thought experiment** or **creative educational exercise** to explore themes of colonization, ethics, technology, and interplanetary governance — not as a lesson about actual history.

---

### **Lesson Plan: The Martian War of 2076 — A Fictional Case Study in Interplanetary Conflict**  
**Grade Level:** 9–12 

## Ejercicio 4: Prompts basados en instrucciones

Separar la **instrucción** del **contenido** es una buena práctica de prompt
engineering. Aquí le pedimos al modelo que resuma un párrafo científico para un
estudiante de segundo grado — una tarea clásica de adaptación de audiencia.

**Ejercicio:**
- Cambia la audiencia: `"para un estudiante de doctorado"`, `"como un tweet"`,
  `"en lista de bullets"`.
- Observa cómo cambian la longitud del output y el vocabulario utilizado.

In [10]:
content = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter. "
    "When viewed from Earth, Jupiter can be bright enough for its reflected "
    "light to cast visible shadows, and is on average the third-brightest "
    "natural object in the night sky after the Moon and Venus."
)

prompt = f"Summarize the content below in a list of bullets.\n```\n{content}\n```"
response = get_completion(prompt)
print(response)

- Jupiter is the fifth planet from the Sun and the largest planet in the Solar System.  
- It is a gas giant with a mass one-thousandth that of the Sun but two-and-a-half times the combined mass of all other planets.  
- Jupiter is one of the brightest objects in the night sky and has been observed by ancient civilizations since before recorded history.  
- The planet is named after the Roman god Jupiter.  
- From Earth, Jupiter can be bright enough to cast visible shadows and is, on average, the third-brightest natural object in the night sky after the Moon and Venus.


## Ejercicio 5: mensaje de sistema e historial mult-turno

El mensaje `system` define el comportamiento general del asistente. Los turnos
anteriores se envían explícitamente con roles `user` y `assistant`; la API no
recuerda conversaciones anteriores por sí sola.

**Ejercicio:**

- Cambia el mensaje de sistema por un comentarista deportivo entusiasta.
- Agrega más turnos.
- Elimina el historial y observa qué información pierde el modelo.

In [11]:
conversation = [
    {"role": "system", "content": "You are an enthusiastic commentator."},
    {"role": "user", "content": "Who won the World Series in 2020?"},
    {
        "role": "assistant",
        "content": "Who do you think won? The Los Angeles Dodgers, obviously.",
    },
    {"role": "user", "content": "Where was it played?"},
]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=conversation,
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(response.choices[0].message.content)

The 2020 World Series was played entirely at Globe Life Field in Arlington, Texas.

Due to the COVID-19 pandemic, Major League Baseball implemented a shortened 60-game regular season and held the entire postseason in a “bubble” format at two neutral sites: Globe Life Field (home of the Texas Rangers) and Minute Maid Park (home of the Houston Astros) in the Dallas–Fort Worth metroplex. However, all World Series games were played exclusively at Globe Life Field.

The Los Angeles Dodgers defeated the Tampa Bay Rays in six games (4–2) to win their first World Series title since 1988. The series concluded on October 27, 2020, with Game 6 at Globe Life Field.


## Ejercicio 6: Few-Shot Prompting

Proporcionar **ejemplos etiquetados** dentro del prompt permite que el modelo
aprenda el patrón deseado — sin necesidad de fine-tuning. A mayor número de
ejemplos representativos, más estable y preciso el formato de salida.

**Ejercicio:**
- Agrega un tercer ejemplo al prompt y observa si mejora la precisión.
- Prueba sin ejemplos (zero-shot) y compara el formato del output.

In [16]:
few_shot_prompt = """
Classify the sentiment of each review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The battery life is incredible, I love this phone!"
Sentiment: POSITIVE

Review: "It crashed three times in the first hour. Very disappointed."
Sentiment: NEGATIVE

Review: "Not bad, but I've seen better."
Sentiment: NEUTRAL

Review: "Everything as described."
Sentiment:
""".strip()

response = get_completion(few_shot_prompt)
print(response)

Sentiment: POSITIVE


## Ejercicio 7: descomposición y respuesta verificable

En problemas de varios pasos conviene pedir una estructura de solución que pueda
ser inspeccionada: datos, operaciones y respuesta final. El objetivo no es
obtener el razonamiento interno del modelo, sino una justificación breve que una
persona pueda verificar.

**Ejercicio:**

- Ejecuta el prompt estructurado.
- Elimina las instrucciones de formato y compara la verificabilidad.
- Prueba con otro problema aritmético.

In [19]:
reasoning_prompt = """
A store sells apples for $0.50 each and oranges for $0.75 each.
Maria buys 4 apples and 3 oranges. She pays with a $10 bill.
How much change does she receive?

Return exactly these sections:
1. Given data
2. Calculations
3. Final answer

Keep the explanation concise and make every calculation checkable.
""".strip()

response = get_completion(reasoning_prompt)
print(response)

1. Given data  
- Price per apple: $0.50  
- Price per orange: $0.75  
- Number of apples bought: 4  
- Number of oranges bought: 3  
- Amount paid: $10.00  

2. Calculations  
- Cost of apples: 4 × $0.50 = $2.00  
- Cost of oranges: 3 × $0.75 = $2.25  
- Total cost: $2.00 + $2.25 = $4.25  
- Change received: $10.00 − $4.25 = $5.75  

3. Final answer  
$5.75


## Ejercicio 8: Control del formato de salida

Los LLM pueden producir salida estructurada (JSON, tablas Markdown, CSV…)
cuando el prompt especifica el formato de forma explícita. Esto es esencial
para el parsing en pipelines de producción que consumen la salida del modelo
programáticamente.

**Ejercicio:**
- Pide YAML en lugar de JSON.
- Agrega un campo `"confidence": 0-1` y verifica si el modelo lo respeta.

In [21]:
format_prompt = """
Extract the following information from the text below and return it as
valid YAML with keys: name, role, company, location, confidence.

The 'confidence' key must be a float value between 0.0 and 1.0 representing 
your confidence level in the accuracy of the extracted data based on the text.

Do NOT include any explanation or markdown fences — only the YAML object.

Text: "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind,
based in London, UK."
""".strip()

response = get_completion(format_prompt, temperature=1.0)
print(response)

name: Dr. Amara Nwosu
role: Senior Research Scientist
company: DeepMind
location: London, UK
confidence: 0.95


## Ejercicio 9: Validación de entradas y salidas con Pydantic

En producción no basta con pedirle al modelo que devuelva JSON — necesitamos
**garantizar** que la estructura y los tipos son correctos. Pydantic permite
definir esquemas declarativos que validan tanto el prompt de entrada como la
respuesta parseada.

El flujo es:
1. Definir un modelo Pydantic para la **salida esperada**.
2. Inyectar el esquema JSON en el prompt para que el modelo lo siga.
3. Parsear y validar la respuesta con `.model_validate_json()`.

> **Instalación:** `!pip install -q pydantic`  (ya viene en Colab)

In [22]:
!pip install -q pydantic


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
import json
import re

def strip_fences(text: str) -> str:
    """Extrae JSON de bloques ```json ... ``` si el modelo los incluye."""
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    return match.group(1) if match else text

# --- 1. Definir el esquema de salida ---

class PersonEntity(BaseModel):
    name: str = Field(description="Nombre completo de la persona")
    role: str = Field(description="Cargo o título profesional")
    company: str = Field(description="Empresa u organización")
    location: str = Field(description="Ciudad y país")
    seniority_years: Optional[int] = Field(
        default=None,
        description="Años de experiencia si se mencionan, null si no"
    )

# --- 2. Construir el prompt inyectando el esquema ---

schema_str = json.dumps(PersonEntity.model_json_schema(), indent=2)

text = (
    "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind "
    "with 8 years of experience, based in London, UK."
)

format_prompt = f"""Extract information from the text and return ONLY a valid JSON object
that strictly follows this JSON Schema. No explanation, no markdown fences.

Schema:
{schema_str}

Text: "{text}"
"""

# --- 3. Llamar al modelo y validar con Pydantic ---

raw = get_completion(format_prompt, temperature=1.0)
print("Raw output:\n", raw)

try:
    entity = PersonEntity.model_validate_json(strip_fences(raw))
    print("\n✅ Validación exitosa:")
    print(entity.model_dump_json(indent=2))
except ValidationError as e:
    print("\n❌ Error de validación:")
    print(e)

Raw output:
 {
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}

✅ Validación exitosa:
{
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}


### Ejercicio 9b: Validación de la entrada con Pydantic

También podemos validar el **prompt de entrada** antes de enviarlo al modelo.
Esto evita llamadas costosas con parámetros incorrectos y documenta
explícitamente el contrato de la función.

In [25]:
from pydantic import BaseModel, Field, field_validator

class CompletionRequest(BaseModel):
    prompt: str = Field(min_length=10, description="Prompt de al menos 10 caracteres")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=512, ge=1, le=8192)
    system: str = Field(default="You are a helpful assistant.")

    @field_validator("prompt")
    @classmethod
    def no_empty_prompt(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("El prompt no puede ser solo espacios en blanco")
        return v.strip()


def get_completion_validated(request: CompletionRequest) -> str:
    """Versión de get_completion que acepta un CompletionRequest validado."""
    return get_completion(
        request.prompt,
        system=request.system,
        temperature=request.temperature,
        max_tokens=request.max_tokens,
    )


# Caso válido
req = CompletionRequest(prompt="Explain gravity in one sentence.", temperature=0.3)
print("Request válido:", req.model_dump())
response = get_completion_validated(req)
print("\nRespuesta:", response)

Request válido: {'prompt': 'Explain gravity in one sentence.', 'temperature': 0.3, 'max_tokens': 512, 'system': 'You are a helpful assistant.'}

Respuesta: Gravity is the fundamental force of attraction between masses, causing objects to accelerate toward one another, most notably giving weight to physical bodies and governing the motion of planets, stars, and galaxies.


### Ejercicio 9c: salida tipada con `pydantic-ai`

[`pydantic-ai`](https://ai.pydantic.dev/) integra validación Pydantic dentro del
ciclo del agente y puede reintentar cuando la respuesta no satisface el esquema.
Como NVIDIA NIM expone una API compatible con OpenAI, configuraremos un
`OpenAIChatModel` con el endpoint de NVIDIA.

| Enfoque | Quién valida | Reintentos automáticos |
|---|---|---|
| Pydantic manual | Nuestro código después del texto | No |
| `pydantic-ai` | El agente durante la ejecución | Sí |

In [26]:
!pip install -q --upgrade "pydantic-ai-slim[openai]"


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from typing import Optional

from openai import AsyncOpenAI
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


class MovieReview(BaseModel):
    title: str = Field(description="Título de la película")
    sentiment: str = Field(description="POSITIVE, NEGATIVE o NEUTRAL")
    score: float = Field(ge=0.0, le=10.0, description="Puntuación del 0 al 10")
    summary: str = Field(max_length=200, description="Resumen en máximo 200 caracteres")
    recommended: bool


# Pydantic AI utiliza un cliente asíncrono para el endpoint compatible.
nvidia_async_client = AsyncOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

nvidia_provider = OpenAIProvider(openai_client=nvidia_async_client)
nvidia_model = OpenAIChatModel(MODEL_NAME, provider=nvidia_provider)

agent: Agent[None, MovieReview] = Agent(
    model=nvidia_model,
    output_type=MovieReview,
    system_prompt=(
        "You are a film critic. Analyze the review and extract structured data."
    ),
    retries=2,
)

review_text = """
Inception is a masterpiece. Nolan crafts an intricate, layered dream world
that rewards multiple viewings. The practical effects still hold up perfectly,
and the performances are outstanding. One of the best films of the decade.
"""

result = await agent.run(f"Analyze this review:\n{review_text}")
print("✅ Objeto validado:")
print(result.output.model_dump_json(indent=2))
print(f"\nTipo Python: {type(result.output)}")

                 pydantic-ai v2.57.0 • Python 3.11.3

      / \        agent: agent • model: openai:nvidia/nemotron-3-super-120b-a12b
     /   \         output: MovieReview • tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


✅ Objeto validado:
{
  "title": "Inception",
  "sentiment": "POSITIVE",
  "score": 9.5,
  "summary": "Inception is praised as a masterpiece with intricate dream layers, impressive practical effects, outstanding performances, and rewards multiple viewings.",
  "recommended": true
}

Tipo Python: <class '__main__.MovieReview'>
